## Raw Data Exploration:

This notebook explores the raw tables loaded from Kaggle and other sources before any transformations are applied. The goal is to understand the structure,
spot data quality issues and document findings that will drive decisions in the next step (02_dim_tables).

FAO datasets:
- raw.Forest_year — forest coverage % per country per year (1990–2025)
- raw.Forest_Policy_Legislation — national/sub-national forest policies and legislation
- raw.Area_protected_by_law_to_reamin_as_forest — protected forest area (1990–2020)
- raw.Naturaly_regenerating_and_primary_forest — naturally regenerating and primary forest (1990–2025)
- raw.Planted_forest — planted forest growing stock (1990–2025)
- raw.Forest_area_change — natural forest expansion per period (1990–2025)
- raw.Forest_change_reason — forest disturbances: fire, insects, diseases (2000–2025)
- raw.forest_purpose — forest designation by purpose (1990–2025)

World Bank / Kaggle datasets:
- raw.GDP — GDP per capita per country per year (1990–2024)
- raw.income_groups — country income group classifications
- raw.Land_Area — country land area in km² (1960–2021)
- raw.Country_mapping — country code mapping (ISO2, ISO3, regions)

Other Kaggle datasets:
- raw.Forest_watch — deforestation and forest disturbance data

## What we are looking for
- Row counts and year ranges per table
- Missing or empty country codes
- Duplicates
- Null values in key columns
- Country code consistency across tables 
- Check which countries in Forest_year have no matching region in the mapping table.

## Findings
- Analysis range: 1990–2025 for forest data, 1990–2024 for GDP (2025 not yet published by World Bank)
- Forest_year has entities without ISO3 codes (EU, England, Scotland etc.) — will be handled with AGG_ prefix in dim_country
- GDP contains one indicator only: GDP per capita (in US$)
- GDP NULLs expected for pre-1990 years — not an issue for our analysis
- 4 countries have forest data but no GDP match: French Guiana, Taiwan, Western Sahara, World — kept in dim_country but excluded from correlation analysis
- No duplicates, no nulls in key columns across Forest_year and GDP
- Taiwan (TWN) and Western Sahara (ESH) have no region — fixed manually in 02_dim_tables
- FAO secondary datasets (Area_protected, Naturaly_regenerating, Planted_forest, Forest_area_change, forest_purpose) are in wide format (years as columns) We will have to transform them intoto long format in 04_raw_data_unpivot

In [9]:
-- Row counts for all raw tables
SELECT 'Forest_year'                                    AS table_name, COUNT(*) AS row_count FROM raw.Forest_year
UNION ALL SELECT 'Forest_watch',                        COUNT(*) FROM raw.Forest_watch
UNION ALL SELECT 'GDP',                                 COUNT(*) FROM raw.GDP
UNION ALL SELECT 'income_groups',                      COUNT(*) FROM raw.income_groups
UNION ALL SELECT 'Land_Area',                           COUNT(*) FROM raw.Land_Area
UNION ALL SELECT 'Country_mapping',                     COUNT(*) FROM raw.Country_mapping
UNION ALL SELECT 'Forest_Policy_Legislation',           COUNT(*) FROM raw.Forest_Policy_Legislation
UNION ALL SELECT 'Area_protected',                      COUNT(*) FROM raw.Area_protected_by_law_to_reamin_as_forest
UNION ALL SELECT 'Naturaly_regenerating',               COUNT(*) FROM raw.Naturaly_regenerating_and_primary_forest
UNION ALL SELECT 'Planted_forest',                      COUNT(*) FROM raw.Planted_forest
UNION ALL SELECT 'Forest_area_change',                  COUNT(*) FROM raw.Forest_area_change
UNION ALL SELECT 'Forest_change_reason',                COUNT(*) FROM raw.Forest_change_reason
UNION ALL SELECT 'forest_purpose',                      COUNT(*) FROM raw.forest_purpose;

(13 rows affected)

table_name                | row_count
--------------------------+----------
Forest_year               | 7970     
Forest_watch              | 5720     
GDP                       | 17290    
stg_gdp_income            | 217      
Land_Area                 | 266      
Country_mapping           | 249      
Forest_Policy_Legislation | 235      
Area_protected            | 238      
Naturaly_regenerating     | 238      
Planted_forest            | 238      
Forest_area_change        | 238      
Forest_change_reason      | 5328     
forest_purpose            | 223      
(13 rows)

Total execution time: 00:00:00.085

In [7]:
-- Year ranges across all tables
SELECT 'Forest_year' AS table_name, 
    MIN(Year) AS year_from, 
    MAX(Year) AS year_to 
FROM raw.Forest_year

UNION ALL

SELECT 'GDP',
    MIN(Year),
    MAX(Year)
FROM raw.GDP;

(2 rows affected)

table_name  | year_from | year_to
------------+-----------+--------
Forest_year | 1000      | 2025   
GDP         | 1960      | 2024   
(2 rows)

Total execution time: 00:00:00.053

In [11]:
-- Sample rows from each table
SELECT TOP 5 * FROM raw.Forest_year;
SELECT TOP 5 * FROM raw.Forest_watch;
SELECT TOP 5 * FROM raw.GDP;
SELECT TOP 5 * FROM raw.income_groups;
SELECT TOP 5 * FROM raw.Land_Area;
SELECT TOP 5 * FROM raw.Country_mapping;
SELECT TOP 5 * FROM raw.Forest_Policy_Legislation;
SELECT TOP 5 * FROM raw.Area_protected_by_law_to_reamin_as_forest;
SELECT TOP 5 * FROM raw.Naturaly_regenerating_and_primary_forest;
SELECT TOP 5 * FROM raw.Planted_forest;
SELECT TOP 5 * FROM raw.Forest_area_change;
SELECT TOP 5 * FROM raw.Forest_change_reason;
SELECT TOP 5 * FROM raw.forest_purpose;


(5 rows affected)
(5 rows affected)
(5 rows affected)
(5 rows affected)
(5 rows affected)
(5 rows affected)
(5 rows affected)
(5 rows affected)
(5 rows affected)
(5 rows affected)
(5 rows affected)
(5 rows affected)
(5 rows affected)

Country     | Year | Forest_percentage | Code | Notes
------------+------+-------------------+------+------
Afghanistan | 1990 | 1,8543152         | AFG  |      
Afghanistan | 1991 | 1,8543152         | AFG  |      
Afghanistan | 1992 | 1,8543152         | AFG  |      
Afghanistan | 1993 | 1,8543152         | AFG  |      
Afghanistan | 1994 | 1,8543152         | AFG  |      
(5 rows)

Country | Year | Forest_Area_km2 | Land_Area_km2 | Forest_Cover_Pct | Annual_Deforestation_Rate | Annual_Afforestation_Rate | Total_Carbon_Stock_Tonnes | Primary_Driver_of_Change
--------+------+-----------------+---------------+------------------+---------------------------+---------------------------+---------------------------+-------------------------
Brazil  | 2000 | 54

In [ ]:
-- Check for missing or empty country codes in Forest_year
SELECT DISTINCT Country, Code
FROM raw.Forest_year
WHERE Code IS NULL OR Code = ''
ORDER BY Country;

(7 rows affected)

Country                       | Code
------------------------------+-----
England                       |     
European Union (27)           |     
High-income countries         |     
Low-income countries          |     
Lower-middle-income countries |     
Scotland                      |     
Upper-middle-income countries |     
(7 rows)

Total execution time: 00:00:01.003

In [ ]:
-- Check for codes longer than 3 characters. To find out if there are different values than country codes, some aggregates.
SELECT DISTINCT Code, Country
FROM raw.Forest_year
WHERE LEN(Code) > 3
ORDER BY Code;

(1 row affected)

Code     | Country
---------+--------
OWID_WRL | World  
(1 row)

Total execution time: 00:00:06.504

In [ ]:
-- Check for duplicate rows in Forest_year
SELECT Code, Country, Year, COUNT(*) AS occurrences
FROM raw.Forest_year
GROUP BY Code, Country, Year
HAVING COUNT(*) > 1;


(0 rows affected)

(0 rows)

Total execution time: 00:00:00.614

In [ ]:
-- Check for nulls in key columns of Forest_year
SELECT
    COUNT(*) - COUNT(Code)              AS null_code,
    COUNT(*) - COUNT(Country)           AS null_country,
    COUNT(*) - COUNT(Year)              AS null_year,
    COUNT(*) - COUNT(Forest_percentage) AS null_forest_pct
FROM raw.Forest_year;

(1 row affected)

null_code | null_country | null_year | null_forest_pct
----------+--------------+-----------+----------------
0         | 0            | 0         | 0              
(1 row)

Total execution time: 00:00:00.987

In [ ]:
-- Check for nulls in key columns of GDP
SELECT
    COUNT(*) - COUNT([Country Name])    AS null_country,
    COUNT(*) - COUNT([Country Code])    AS null_country_code,
    COUNT(*) - COUNT(Year)              AS null_year,
    COUNT(*) - COUNT(GDP)               AS null_gdp
FROM raw.GDP;

(1 row affected)

null_country | null_country_code | null_year | null_gdp
-------------+-------------------+-----------+---------
0            | 0                 | 0         | 2729    
(1 row)

Total execution time: 00:00:00.942

In [ ]:
-- Check if country codes in Forest_year exist in GDP
SELECT DISTINCT f.Code, f.Country
FROM raw.Forest_year f
LEFT JOIN raw.GDP g ON g.[Country Code] = f.Code
WHERE g.[Country Code] IS NULL
AND f.Code IS NOT NULL
AND f.Code <> ''
ORDER BY f.Country;

(4 rows affected)

Code     | Country       
---------+---------------
GUF      | French Guiana 
TWN      | Taiwan        
ESH      | Western Sahara
OWID_WRL | World         
(4 rows)

Total execution time: 00:00:00.744

In [5]:
-- Check country code match between Forest_year and Country_mapping
SELECT DISTINCT f.Code, f.Country
FROM raw.Forest_year f
LEFT JOIN raw.Country_mapping m ON m.[alpha-3] = f.Code
WHERE m.[alpha-3] IS NULL
    AND f.Code IS NOT NULL
    AND f.Code != ''
ORDER BY f.Country;

(1 row affected)

Code     | Country
---------+--------
OWID_WRL | World  
(1 row)

Total execution time: 00:00:00.075